<a href="https://colab.research.google.com/github/simply-logical/mlbook_ii_notebooks/blob/master/Data_Visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install datasets pyarrow mlcroissant --quiet

In [ ]:
import matplotlib.pyplot as plt
from abc import ABC, abstractmethod

import re
import os
import tarfile
import requests
from collections import Counter

import pandas as pd
import seaborn as sns
from mlcroissant import Dataset

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize

In [ ]:
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords')

In [ ]:
import warnings
import logging
from absl import logging as absl_logging

warnings.filterwarnings("ignore")
absl_logging.set_verbosity(absl_logging.ERROR)
logging.getLogger().setLevel(logging.ERROR)

In [ ]:
class DataSource(ABC):
  @abstractmethod
  def load_data(self) -> pd.DataFrame:
    pass

In [ ]:
class EuroparlDataset(DataSource):
  def __init__(self, language_pair: str="pt-en", save_path: str="data/europarl"):

    self.language_pair = language_pair
    self.save_path = save_path
    self.url = f"https://www.statmt.org/europarl/v7/{language_pair}.tgz"
    self.filename = f"{language_pair}.tgz"
    self.full_path = os.path.join(self.save_path, self.filename)

  def _ensure_directory(self):

    if not os.path.exists(self.save_path):
      os.makedirs(self.save_path)

  def _download(self):

    self._ensure_directory()
    if not os.path.exists(self.full_path):
      print(f"Downloading {self.url}...")
      response = requests.get(self.url, stream=True)

      if response.status_code == 200:
        with open(self.full_path, "wb") as f:
          for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
        print(f"Download of Europarl {self.language_pair} finished.")
      else:
        print(f"Failed to download {url}. Status code: {response.status_code}")

  def _extract_and_load(self) -> pd.DataFrame:

      dataframe = pd.DataFrame()

      with tarfile.open(self.full_path, "r:gz") as tar:
        files = tar.getnames()
        for target_file in files:

          print(f" Extracting: {target_file}")
          f = tar.extractfile(target_file)
          content = f.read().decode("utf-8").splitlines()[:1000]
          dataframe[f'{target_file[-2:]}'] = content

      return dataframe

  def load_data(self)-> pd.DataFrame:

    self._download()
    return self._extract_and_load()




In [ ]:
class EuroparlDataset(DataSource):
  def __init__(self, languages: list=['pt','en','es'], save_path: str="data/europarl"):

    self.languages = [language for language in languages if language != 'en']
    self.save_path = save_path
    self.urls = [f"https://www.statmt.org/europarl/v7/{language}-en.tgz" for language in self.languages]
    self.filenames = [f"{language}.tgz" for language in self.languages]
    self.full_paths = [os.path.join(self.save_path, filename) for filename in self.filenames]

  def _ensure_directory(self):

    if not os.path.exists(self.save_path):
      os.makedirs(self.save_path)

  def _download(self):

    for url, full_path in zip(self.urls, self.full_paths):

      self._ensure_directory()
      if not os.path.exists(full_path):
        print(f"Downloading {url}...")
        response = requests.get(url, stream=True)
        with open(full_path, "wb") as f:
          for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
        print(f"Download of Europarl file finished.")

  def _extract_and_load(self) -> pd.DataFrame:

      dataframes = [pd.DataFrame(), pd.DataFrame()]

      for idx, full_path in enumerate(self.full_paths):

        with tarfile.open(full_path, "r:gz") as tar:
          files = tar.getnames()
          for target_file in files:

            column = target_file[-2:]

            print(f" Extracting: {target_file}")
            f = tar.extractfile(target_file)
            content = f.read().decode("utf-8").splitlines()[:1000]
            dataframes[idx][f'{column}'] = content

      dataframe = pd.merge(dataframes[0], dataframes[1], on='en')


      return dataframe

  def load_data(self)-> pd.DataFrame:

    self._download()
    return self._extract_and_load()




In [ ]:
def _remove_punctuation(text: str) -> str:
  return re.sub(r'[^\w\s]', '', text)

In [ ]:
class HelloSimpleAIDataset(DataSource):

  def __init__(self, source="wiki_csai"):
    self.url = "https://huggingface.co/api/datasets/Hello-SimpleAI/HC3/croissant"
    self.source = source

  def load_data(self) -> pd.DataFrame:

    ds = Dataset(jsonld=self.url)
    records_iterator = ds.records(self.source)

    data = []
    print(f"Processing Hello-SimpleAI - {self.source}")
    data = [item for item in records_iterator]

    print(f"Done.Total records: {len(data)}")
    return pd.DataFrame(data)

In [ ]:
class DataCleaner(ABC):
  @abstractmethod
  def clean_data(self, data: pd.DataFrame) -> pd.DataFrame:
    pass

In [ ]:
class EuroparlDataCleaner(DataCleaner):
  def __init__(self, languages: list=['pt', 'en']):
    self.languages = languages

  def _count_words_starting_uppercase(self, text: str)  -> int:
    words = text.split()
    count = sum(1 for word in words if word[0].isupper())
    return count

  def _remove_punctuation(self, text: str) -> str:
    return re.sub(r'[^\w\s]', '', text)

  def clean_data(self, data: pd.DataFrame) -> pd.DataFrame:

    cleaned_dataset = pd.DataFrame()
    for column in data.columns:
      cleaned_dataset[column] = data[column].apply(lambda text: str.lower(str(text)))

    return cleaned_dataset

In [ ]:
class HelloSimpleAIDataCleaner(DataCleaner):
  def __init__(self, source="wiki_csai"):
    self.source = source

  def _clean_text(self, raw_text):
      text = re.sub(r"^\[?b['\"]|['\"]\]?$", "", raw_text)

      text = re.sub(r'\[[^\]]+\]', '', text)

      text = re.sub(r'\b(e\.g\.)', 'eg', text, flags=re.IGNORECASE)

      text = re.sub(r'\b(i\.e\.)', 'ie', text, flags=re.IGNORECASE)

      text = re.sub(r'\{\\displaystyle .*?\}', '', text)

      text = re.sub(r'\\x[0-9a-fA-F]{2}', '', text)

      text = re.sub(r'(\\n)+', ' ', text)

      text = re.sub(r'\\', '', text)

      text = re.sub(r"\s+", " ", text).strip()
      text = text.lower()

      return text

  def clean_data(self, data: pd.DataFrame) -> pd.DataFrame:

    for column in data.columns:
      data[column] = data[column].apply(lambda text: self._clean_text(str(text)))

    return data


In [ ]:
class DataProcessor(ABC):
  @abstractmethod
  def process_dataframe(self, data: pd.DataFrame) -> pd.DataFrame:
    pass

In [ ]:
class EuroparlDataProcessor(DataProcessor):
  def __init__(self, languages: list=['pt', 'en', 'es']):
    self.languages = languages

  def _remove_punctuation(self, text: str) -> str:
    return re.sub(r'[^\w\s]', '', text)

  def _count_words_starting_uppercase(self, text: str)  -> int:
    words = text.split()
    count = sum(1 for word in words if word[0].isupper())
    return count

  def process_dataframe(self, df: pd.DataFrame, text_column: str) -> pd.DataFrame:
        print(f'Processing column: {text_column}')
        df['sentence_list'] = df[text_column].apply(sent_tokenize)

        df_sentences = df.explode('sentence_list').reset_index(drop=True)
        df_sentences = df_sentences.rename(columns={'sentence_list': 'sentence'})


        df_sentences = df_sentences[df_sentences['sentence'].str.strip().astype(bool)]

        df_sentences = df_sentences.dropna(subset=['sentence'])


        df_sentences['num_uppercase_words'] = df_sentences['sentence'].apply(self._count_words_starting_uppercase)
        df_sentences['sentence'] = df_sentences['sentence'].apply(lambda text: str.lower(str(text)))
        df_sentences['tokens'] = df_sentences['sentence'].astype(str).apply(self._remove_punctuation).apply(word_tokenize)
        df_sentences['word_count'] = df_sentences['tokens'].apply(len)
        df_sentences['uppercase_frequency'] = df_sentences['num_uppercase_words'] / df_sentences['word_count']

        df_sentences['avg_word_len'] = df_sentences['tokens'].apply(
            lambda words: sum(len(w) for w in words) / len(words) if len(words) > 0 else 0
        )

        df_sentences['char_count'] = df_sentences['sentence'].apply(len)

        return df_sentences

In [ ]:
class HelloSimpleAIDataProcessor(DataProcessor):
  def __init__(self, source="wiki_csai"):
    self.source = source

  def _remove_punctuation(self, text: str) -> str:
    return re.sub(r'[^\w\s]', '', text)

  def process_dataframe(self, df: pd.DataFrame, text_column: str) -> pd.DataFrame:
    print(f'Processing column: {text_column}')

    df['sentence_list'] = df[text_column].apply(sent_tokenize)

    df_sentences = df.explode('sentence_list').reset_index(drop=True)
    df_sentences = df_sentences.rename(columns={'sentence_list': 'sentence'})

    df_sentences = df_sentences[df_sentences['sentence'].str.strip().astype(bool)]

    df_sentences = df_sentences.dropna(subset=['sentence'])

    df_sentences['tokens'] = df_sentences['sentence'].astype(str).apply(self._remove_punctuation).apply(word_tokenize)
    df_sentences['word_count'] = df_sentences['tokens'].apply(len)

    df_sentences['avg_word_len'] = df_sentences['tokens'].apply(
            lambda words: sum(len(w) for w in words) / len(words) if len(words) > 0 else 0
    )

    df_sentences['char_count'] = df_sentences['sentence'].apply(len)

    return df_sentences

In [ ]:
class HelloSimpleAIDataVisualizer():
    def __init__(self, palette: list=["#3498db", "#e67e22", "#9b59b6"], aplha: float=0.5):
        self.palette = palette
        self.alpha = aplha

    def plot_bar_chart(self, df: pd.DataFrame, title: str):
      plt.figure(figsize=(15, 6))
      ax = sns.barplot(data=df, palette=self.palette, alpha=self.alpha)
      for container in ax.containers:
          ax.bar_label(container, fmt='%.2f', padding=3)

      plt.title(title)
      plt.grid(True, axis='y', linestyle='--', alpha=0.7)

      sns.despine()

      plt.show()
      plt.close()

    def plot_histogram(self, df:  pd.DataFrame, title: str):


      for column, color in zip(df.columns, self.palette):

        plt.figure(figsize=(14.625,6))

        sns.histplot(
            df[column],
            kde=False,
            color=color,
            label=column,
            alpha=self.alpha,
            edgecolor="white",
            shrink=0.8,
            linewidth=0.25
        )

        plt.title(title)
        plt.xlabel(column)
        plt.ylabel("Count")
        plt.legend()

        sns.despine()
        plt.grid(axis='y', alpha=0.3)
        plt.show()
        plt.close()


        plt.show()
        plt.close()

    def plot_scatter_plot(self,
                          df1: pd.DataFrame,
                          df2: pd.DataFrame,
                          title: str,
                          label1: str,
                          label2: str,
                          column1: str='avg_word_len',
                          column2: str='char_count'
                          ):
        plt.figure(figsize=(12.165, 6))


        sns.scatterplot(
            data=df1[[column1, column2]],
            x=column2,
            y=column1,
            label=label1,
            alpha=0.7,
            edgecolor=None,
            color=self.palette[0]
        )


        sns.scatterplot(
            data=df2[[column1, column2]],
            x=column2,
            y=column1,
            label=label2,
            alpha=0.3,
            edgecolor=None,
            color=self.palette[1]
        )

        plt.grid(axis='y', alpha=0.3)
        plt.title(title, fontsize=16, pad=20)
        plt.xlabel("Sentence Length", fontsize=12)
        plt.ylabel("Average Word Length", fontsize=12)


        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)

        sns.despine()
        plt.tight_layout()
        plt.show()

    def plot_boxplot(self, df: pd.DataFrame,  title: str='Word Count'):

        plt.figure(figsize=(14.2, 6))
        sns.boxplot(df, palette=self.palette, boxprops=dict(alpha=self.alpha), orient='v', )
        sns.despine()

        plt.title(title, fontsize=16, pad=20)
        plt.grid(True, axis='y', linestyle='--', alpha=0.7)
        plt.xlabel("Language", fontsize=12)
        plt.ylabel(title, fontsize=12)

        plt.show()

    def plot_most_frequent_words(self, df: pd.DataFrame, title:str, top_n: int=10):

        plt.figure(figsize=(12.165, 6))

        ax = sns.barplot(x=df['words'], y=df['count'],  palette=["#3498db"])
        for container in ax.containers:
          ax.bar_label(container, fmt='%.3f', padding=3)

        plt.title(title, fontsize=18, pad=20)
        plt.grid(True, axis='y', linestyle='--', alpha=0.7)
        sns.despine()
        plt.xlabel('Words', fontsize=14)
        plt.ylabel('Frequency', fontsize=14)

        plt.tight_layout()
        plt.show()



In [ ]:
class EuropalDataVisualizer():
    def __init__(self, palette: list=["#3498db", "#e67e22", "#9b59b6"], aplha: float=0.5):
        self.palette = palette
        self.alpha = aplha

    def plot_bar_chart(self, df: pd.DataFrame, title: str):
      plt.figure(figsize=(15, 6))
      ax = sns.barplot(data=df, palette=self.palette, alpha=self.alpha)
      for container in ax.containers:
          ax.bar_label(container, fmt='%.2f', padding=3)

      plt.title(title)
      plt.grid(True, axis='y', linestyle='--', alpha=0.7)

      sns.despine()

      plt.show()
      plt.close()

    def plot_histogram(self, df:  pd.DataFrame, title: str):


      for column, color in zip(df.columns, self.palette):

        plt.figure(figsize=(14.625,6))

        sns.histplot(
            df[column],
            kde=False,
            color=color,
            label=column,
            alpha=self.alpha,
            edgecolor="white",
            shrink=0.8,
            linewidth=0.25
        )

        plt.title(title)
        plt.xlabel(column)
        plt.ylabel("Count")
        plt.legend()

        sns.despine()
        plt.grid(axis='y', alpha=0.3)
        plt.show()
        plt.close()


        plt.show()
        plt.close()

    def plot_scatter_plot(self,
                          df1: pd.DataFrame,
                          df2: pd.DataFrame,
                          df3: pd.DataFrame,
                          title: str,
                          label1: str,
                          label2: str,
                          label3,
                          column1: str='avg_word_len',
                          column2: str='char_count'
                          ):
        plt.figure(figsize=(12.165, 6))


        sns.scatterplot(
            data=df1[[column1, column2]],
            x=column2,
            y=column1,
            label=label1,
            alpha=0.7,
            edgecolor=None,
            color=self.palette[0]
        )


        sns.scatterplot(
            data=df2[[column1, column2]],
            x=column2,
            y=column1,
            label=label2,
            alpha=0.3,
            edgecolor=None,
            color=self.palette[1]
        )

        sns.scatterplot(
            data=df3[[column1, column2]],
            x=column2,
            y=column1,
            label=label3,
            alpha=0.3,
            edgecolor=None,
            color=self.palette[2]
        )

        plt.grid(axis='y', alpha=0.3)
        plt.title(title, fontsize=16, pad=20)
        plt.xlabel("Sentence Length", fontsize=12)
        plt.ylabel("Average Word Length", fontsize=12)


        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)

        sns.despine()
        plt.tight_layout()
        plt.show()

    def plot_boxplot(self, df: pd.DataFrame,  title: str='Word Count'):

        plt.figure(figsize=(14.2, 6))
        sns.boxplot(df, palette=self.palette, boxprops=dict(alpha=self.alpha), orient='v', )
        sns.despine()

        plt.title(title, fontsize=16, pad=20)
        plt.grid(True, axis='y', linestyle='--', alpha=0.7)
        plt.xlabel("Language", fontsize=12)
        plt.ylabel(title, fontsize=12)

        plt.show()

    def plot_most_frequent_words(self, df: pd.DataFrame, title:str, top_n: int=10):

        plt.figure(figsize=(12.165, 6))

        ax = sns.barplot(x=df['words'], y=df['count'],  palette=["#3498db"])
        for container in ax.containers:
          ax.bar_label(container, fmt='%.3f', padding=3)

        plt.title(title, fontsize=18, pad=20)
        plt.grid(True, axis='y', linestyle='--', alpha=0.7)
        sns.despine()
        plt.xlabel('Words', fontsize=14)
        plt.ylabel('Frequency', fontsize=14)

        plt.tight_layout()
        plt.show()



In [ ]:
def most_frequent_words(df: pd.DataFrame, use_stop_words: bool=True, top_n: int=10):

    all_words = df['tokens'].explode()
    all_words = all_words.str.lower().dropna()

    if not use_stop_words:
      stop_words = set(nltk.corpus.stopwords.words('english'))
      all_words = all_words[~all_words.isin(stop_words)]

    word_counts = Counter(all_words)

    return pd.DataFrame(list(word_counts.most_common(top_n)), columns=['words', 'count'])

In [ ]:
class Classification(ABC):
  @abstractmethod
  def run_classification(self):
    pass

In [ ]:
import numpy as np

from collections import Counter
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

class HelloSimpleAIClassification(Classification):
    def __init__(self, df1: pd.DataFrame, df2: pd.DataFrame, vocab_size: int, source: str="wiki_csai" ):
      self.source = source
      self.df_human = df1
      self.df_gpt = df2
      self.vocab_size = vocab_size
      self.vocab = self._vocab_to_dict(self._get_one_hot_vocab())
      self.embeddings = self._generate_one_hot_embeddings()
      self.tf_idf_vector = self._generate_tf_idf_vector()
      self.target = self._generate_target()
      self.model = LogisticRegression(max_iter=1000)

    def _most_frequent_words(self, df: pd.DataFrame, use_stop_words: bool=True, top_n: int=10):

      all_words = df['tokens'].explode()
      all_words = all_words.str.lower().dropna()

      if not use_stop_words:
        stop_words = set(nltk.corpus.stopwords.words('english'))
        all_words = all_words[~all_words.isin(stop_words)]

      word_counts = Counter(all_words)

      return pd.DataFrame(list(word_counts.most_common(top_n)), columns=['words', 'count'])

    def _get_one_hot_vocab(self):
      size = self.vocab_size*2
      top_words_human = self._most_frequent_words(self.df_human, use_stop_words=False, top_n=size)
      top_words_gpt = self._most_frequent_words(self.df_gpt, use_stop_words=False, top_n=size)

      union = pd.Series(np.union1d(top_words_human['words'], top_words_gpt['words']))
      intersection = pd.Series(np.intersect1d(top_words_human['words'], top_words_gpt['words']))

      abs_dif = union[~union.isin(intersection)]

      if abs_dif.size > self.vocab_size:
        abs_dif = abs_dif[:self.vocab_size]

      if abs_dif.size < self.vocab_size:
        complement = np.random.choice(intersection, self.vocab_size - abs_dif.size, replace=False)
        abs_dif = np.concatenate((abs_dif, complement))

      return list(abs_dif)

    def _vocab_to_dict(self, vocab: list):
      vocab_dict = {}
      for i, word in enumerate(vocab):
        vocab_dict[word] = i
      return vocab_dict

    def _generate_one_hot_embeddings(self):
      df = pd.concat([self.df_human, self.df_gpt])
      one_hot_embeddings = np.zeros((len(df), self.vocab_size))

      for i, tokens in enumerate(df['tokens']):
        for token in tokens:
          if token in self.vocab:
            one_hot_embeddings[i, self.vocab[token]] = 1

      return one_hot_embeddings

    def _generate_target(self):
      return np.concatenate((np.zeros(len(self.df_human)), np.ones(len(self.df_gpt))))

    def get_target(self):
      return self.target

    def get_one_hot_embeddings(self):
      return self.embeddings

    def get_vocab(self):
      return self.vocab

    def get_df(self):
      return pd.concat([self.df_human, self.df_gpt])

    def view_sentences_and_embeddings(self):
      concat_df = pd.concat([self.df_human, self.df_gpt])

      concat_df['embeddings'] = list(self.embeddings)

      return concat_df[['tokens', 'embeddings']]

    def _plot_score_graphics(self, word_relevance: pd.DataFrame,  title: str, top_n: int=10):

      plt.figure(figsize=(12.165, 6))
      plt.title(title, fontsize=18, pad=20)

      ax = sns.barplot(x=word_relevance['word'], y=word_relevance['coefficient'],  palette=["#3498db"])
      for container in ax.containers:
          ax.bar_label(container, fmt='%.2f', padding=3)

      plt.grid(True, axis='y', linestyle='--', alpha=0.7)

      sns.despine()

      plt.show()
      plt.close()

    def _plot_decision_boundaries(self, word_relevance: pd.DataFrame, inverted_vocab: dict, coefs: list, top_n: int):

      df_plot = pd.DataFrame({
        'word': [inverted_vocab[i] for i in range(len(coefs))],
        'coefficients': coefs,
        'abs_coef': np.abs(coefs)
      })

      df_plot = df_plot.sort_values(by='coefficients')

      plt.figure(figsize=(12.465, 6))

      y_jitter = np.random.normal(0, 0.1, size=len(df_plot))

      scatter = plt.scatter(
          df_plot['coefficients'],
          y_jitter,
          alpha=0.4,
          c=df_plot['coefficients'],
          cmap='coolwarm',
          s=df_plot['abs_coef'] * 500
      )


      top_gpt = df_plot.tail(top_n)
      for i, row in top_gpt.iterrows():
          plt.annotate(row['word'], (row['coefficients'], y_jitter[df_plot.index.get_loc(i)]),
                      xytext=(5, 5), textcoords='offset points', fontsize=12, fontweight='bold')

      top_human = df_plot.head(top_n)
      for i, row in top_human.iterrows():
          plt.annotate(row['word'], (row['coefficients'], y_jitter[df_plot.index.get_loc(i)]),
                      xytext=(-5, 5), textcoords='offset points', fontsize=12, fontweight='bold', ha='right')

      plt.axvline(0, color='black', linestyle='--', alpha=0.6)
      plt.title('Weight Distribution For Words', fontsize=16)
      plt.xlabel('Coefficient (Negative: Human | Positive: GPT)', fontsize=14)
      plt.yticks([])
      plt.gca().spines['left'].set_visible(False)
      plt.gca().spines['right'].set_visible(False)
      plt.gca().spines['top'].set_visible(False)

      plt.colorbar(scatter, label='Coefficient Intensity')
      plt.show()


    def _plot_logistic_curve(self):

      probs = self.model.predict_proba(self.embeddings)[:, 1]
      log_odds = self.model.decision_function(self.embeddings)

      df_curve = pd.DataFrame({
          'log_odds': log_odds,
          'probs': probs,
          'target': self.target
      }).sort_values(by='log_odds')

      plt.figure(figsize=(12.165, 6))

      plt.scatter(df_curve['log_odds'], df_curve['target'], alpha=0.3, c=df_curve['target'], cmap='coolwarm', label='Real data (0: Human, 1: GPT)')
      plt.plot(df_curve['log_odds'], df_curve['probs'], color='black', linewidth=2, label='Sigmoid curve')

      plt.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
      plt.axvline(0, color='gray', linestyle='--', alpha=0.5)

      plt.title('Logistic Curve (Probs vs. Log-Odds)', fontsize=14)
      plt.xlabel('Linear Combination of Weights (Decision Function)', fontsize=12)
      plt.ylabel('Predicted Probabilities (GPT)', fontsize=12)
      plt.legend()
      plt.grid(True, alpha=0.3)
      plt.show()



    def _score_interpretation(self, top_n: int=10):

      inv_vocab = {v: k for k, v in self.vocab.items()}
      coefs = self.model.coef_[0]

      word_relevance = pd.DataFrame({
          'word': [inv_vocab[i] for i in range(len(coefs))],
          'coefficient': coefs
      })

      gpt_words = word_relevance.sort_values(by='coefficient', ascending=False).head(top_n)
      human_words = word_relevance.sort_values(by='coefficient', ascending=True).head(top_n)

      print(f"\n--- Top {top_n} words for GPT (Class 1) ---")
      print(gpt_words)

      self._plot_score_graphics(gpt_words, f"Top {top_n} words for GPT (Class 1)", top_n)

      print(f"\n--- Top {top_n} words for Humans (Class 0) ---")
      print(human_words)

      self._plot_score_graphics(human_words,  f"Top {top_n} words for Human (Class 0)", top_n)

      self._plot_decision_boundaries(word_relevance, inv_vocab, coefs, top_n)
      self._plot_logistic_curve()


    def run_classification(self):

      X_train, X_test, y_train, y_test = train_test_split(
          self.embeddings,
          self.target,
          test_size=0.2,
          random_state=42,
          shuffle=True,
          stratify=self.target
      )

      print('Training Logistic Regression...')
      self.model.fit(X_train, y_train)

      y_pred = self.model.predict(X_test)
      accuracy = accuracy_score(y_test, y_pred)
      print(f"Accuracy: {accuracy:.2%}")

      self._score_interpretation()

    def _generate_tf_idf_vector(self):

      df = pd.concat([self.df_gpt, self.df_human])

      tf_idf_vector = TfidfVectorizer()

      return tf_idf_vector.fit_transform(df['sentence'])


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import PartialDependenceDisplay
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import pandas as pd


class EuroparlClassification(Classification):

    def __init__(
        self,
        df1: pd.DataFrame,
        df2: pd.DataFrame,
        df3: pd.DataFrame,
        languages: list = ['en', 'pt', 'es']
    ):

        self.languages = languages

        self.df1 = df1
        self.df2 = df2
        self.df3 = df3
        self.model = RandomForestClassifier(
            n_estimators=200,
            max_depth=8,
            random_state=42,
            n_jobs=-1
        )
        self.dataset = self._process_datasets()

    def _process_datasets(self):

      datasets = [self.df1, self.df2, self.df3]

      processed = []

      for idx, dataset in enumerate(datasets):

          temp = dataset[
              ['num_uppercase_words', 'word_count']
          ].copy()

          temp['label'] = idx

          processed.append(temp)

      classification_dataset = pd.concat(
          processed,
          ignore_index=True
      )

      classification_dataset['label'] = (
          classification_dataset['label']
          .astype(int)
      )

      return classification_dataset

    def _partial_dependency_plots(self):

      features = [
          ('num_uppercase_words', 'word_count')
      ]

      for class_idx, language in enumerate(self.languages):

          fig, ax = plt.subplots(
              figsize=(8, 6)
          )

          display = PartialDependenceDisplay.from_estimator(
              estimator=self.model,
              X=self.dataset[
                  ['num_uppercase_words', 'word_count']
              ],
              features=features,
              target=class_idx,
              kind='average',
              grid_resolution=50,
              ax=ax
          )

          for axis in display.axes_.ravel():
              axis.grid(True)

          plt.suptitle(
              f'2D PDP - {language.upper()}'
          )

          plt.tight_layout()
          plt.show()

    def run_classification(self):

        X = self.dataset[
            ['num_uppercase_words', 'word_count']
        ]

        y = self.dataset['label']

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.2,
            random_state=42,
            shuffle=True,
            stratify=y
        )

        print('Training Random Forest...')

        self.model.fit(X_train, y_train)

        y_pred = self.model.predict(X_test)

        accuracy = accuracy_score(
            y_test,
            y_pred
        )

        print(f'Accuracy: {accuracy:.2%}')

        self._partial_dependency_plots()

In [ ]:
languages=['pt', 'en', 'de']
dataset = EuroparlDataset(languages=['pt', 'en', 'de'])
dataset = dataset.load_data()

data_cleaner = EuroparlDataCleaner()

data_processor = EuroparlDataProcessor(languages=languages)

processed_data_1 = data_processor.process_dataframe(dataset, languages[0])
processed_data_2 = data_processor.process_dataframe(dataset, languages[1])
processed_data_3 = data_processor.process_dataframe(dataset, languages[2])

print(processed_data_1.columns)

In [ ]:
classification = EuroparlClassification(processed_data_1, processed_data_2, processed_data_3)
classification.run_classification()

In [ ]:
processed_data_1.shape[0]

In [ ]:
processed_data_1.iloc[-1]

In [ ]:
processed_data_1.index

In [ ]:
class analysis(ABC):
  @abstractmethod
  def run_analysis(self):
    pass

In [ ]:
class EuroparlAnalysis(analysis):
  def __init__(self, languages: list=['pt', 'en', 'es']):
    self.languages = languages

  def run_analysis(self):
    dataset = EuroparlDataset(languages=self.languages)
    dataset = dataset.load_data()

    data_cleaner = EuroparlDataCleaner()
    #cleaned_data = data_cleaner.clean_data(dataset)

    data_processor = EuroparlDataProcessor(languages=self.languages)

    processed_data_1 = data_processor.process_dataframe(dataset, self.languages[0])
    processed_data_2 = data_processor.process_dataframe(dataset, self.languages[1])
    processed_data_3 = data_processor.process_dataframe(dataset, self.languages[2])
    print(processed_data_1.head())
    sentence_len = pd.DataFrame({
            self.languages[0]: processed_data_1['word_count'],
            self.languages[1]: processed_data_2['word_count'],
            self.languages[2]: processed_data_2['word_count'],
    })

    data_visualizer = EuropalDataVisualizer()
    data_visualizer.plot_bar_chart(df=sentence_len.mean(), title="Average sentence length")

    data_visualizer.plot_boxplot(sentence_len, title="Sentence Length Boxplot")
    data_visualizer.plot_histogram(sentence_len, f"Sentence length distribution")

    data_visualizer.plot_scatter_plot(processed_data_1, processed_data_2, processed_data_3, title=f"Sentence Length (# characters) x Avg Word Length", label1=self.languages[0], label2=self.languages[1], label3=self.languages[2])
    data_visualizer.plot_scatter_plot(processed_data_1, processed_data_2, processed_data_3, title=f"Sentence Length (# words) x Avg Word Length", label1=self.languages[0], label2=self.languages[1], label3=self.languages[2], column1='avg_word_len', column2='word_count')
    data_visualizer.plot_scatter_plot(processed_data_1, processed_data_2, processed_data_3, title=f"Uppercase count x Word Count ", label1=self.languages[0], label2=self.languages[1], label3=self.languages[2], column1='num_uppercase_words', column2='word_count')
    data_visualizer.plot_scatter_plot(processed_data_1, processed_data_2, processed_data_3, title=f"Uppercase frequency x Word Count ", label1=self.languages[0], label2=self.languages[1], label3=self.languages[2], column1='uppercase_frequency', column2='word_count')


In [ ]:
class HelloSimpleAIAnalysis(analysis):
  def __init__(self, source="wiki_csai"):
    self.source = source

  def run_analysis(self):
    dataset = HelloSimpleAIDataset(source=self.source)
    dataset = dataset.load_data()

    data_cleaner = HelloSimpleAIDataCleaner()
    cleaned_data = data_cleaner.clean_data(dataset)

    data_processor = HelloSimpleAIDataProcessor(source=self.source)
    processed_data_human = data_processor.process_dataframe(cleaned_data, f'{self.source}/human_answers')
    processed_data_gpt = data_processor.process_dataframe(cleaned_data, f'{self.source}/chatgpt_answers')

    sentence_len = pd.DataFrame({
      'human': processed_data_human['word_count'],
      'gpt': processed_data_gpt['word_count']
    })

    data_visualizer = HelloSimpleAIDataVisualizer()
    data_visualizer.plot_bar_chart(df=sentence_len.mean(), title="Average sentence length")

    data_visualizer.plot_boxplot(sentence_len, title="Sentence Length Boxplot")
    data_visualizer.plot_histogram(sentence_len, f"Sentence length distribution")

    data_visualizer.plot_scatter_plot(processed_data_human, processed_data_gpt, title=f"Sentence Length (# characters) x Avg Word Length", label1='human', label2='gpt')
    data_visualizer.plot_scatter_plot(processed_data_human, processed_data_gpt, title=f"Sentence Length (# words) x Avg Word Length", label1='human', label2='gpt', column1='avg_word_len', column2='word_count')

    data_visualizer.plot_most_frequent_words(most_frequent_words(processed_data_human, use_stop_words=True), f"Most frequent words with stop words (Human) - {self.source}")
    data_visualizer.plot_most_frequent_words(most_frequent_words(processed_data_gpt, use_stop_words=True), f"Most frequent words with stop words (GPT) - {self.source}")

    data_visualizer.plot_most_frequent_words(most_frequent_words(processed_data_human, use_stop_words=False), f"Most frequent words without stop words (Human) - {self.source}")
    data_visualizer.plot_most_frequent_words(most_frequent_words(processed_data_gpt, use_stop_words=False), f"Most frequent words without stop words (GPT) - {self.source}")

    classification = HelloSimpleAIClassification(processed_data_human, processed_data_gpt, vocab_size=20)
    classification.run_classification()

In [ ]:

import ipywidgets as widgets
from IPython.display import display, clear_output


dropdown = widgets.Dropdown(
    options=["HelloSimple-AI", "Europarl"],
    description='Source:',
    layout=widgets.Layout(width='auto')
)

lang_select = widgets.SelectMultiple(
    options=['en','pt', 'es', 'fr', 'da', 'de', 'fi','it', 'el', 'et', 'nl'],
    value=['pt'],
    description='Languages:',
    rows=5,
    layout=widgets.Layout(width='auto', display='none'),
    style={'description_width': 'initial'}
)

lang_help = widgets.HTML(
    value="<small>Hold <b>Ctrl</b> (Win) or <b>Cmd</b> (Mac) to select 3 itens</small>",
    layout=widgets.Layout(display='none', margin='-10px 0 0 85px')
)

source_dropdown = widgets.Dropdown(
    options=['wiki_csai', 'all_splits', 'all', 'finance_splits', 'finance', 'medicine_splits', 'medicine', 'open_qa_splits', 'open_qa', 'reddit_eli5_splits', 'reddit_eli5', 'wiki_csai_splits'],
    description='Sub-Source:',
    layout=widgets.Layout(width='auto', display='flex')
)

button = widgets.Button(
    description="Plot",
    button_style='info',
    layout=widgets.Layout(width='auto', height='40px')
)

output = widgets.Output()

def on_source_change(change):
    if change['new'] == "Europarl":
        lang_select.layout.display = 'flex'
        lang_help.layout.display = 'flex'
        source_dropdown.layout.display = 'none'
    elif change['new'] == "HelloSimple-AI":
        lang_select.layout.display = 'none'
        lang_help.layout.display = 'none'
        source_dropdown.layout.display = 'flex'

def on_click_action(b):
    with output:
        clear_output()
        selected = lang_select.value

        if dropdown.value == "Europarl":
            if len(selected) != 3:
                print(f"Error: Select exctly 3 languages. (You selected {len(selected)})")
                return
            else:
                print(f"Generating plot for: {', '.join(selected)}")
                print(f"Generating plot for Europarl: {source_dropdown.value}")
                analysis = EuroparlAnalysis(languages=selected)
                analysis.run_analysis()

        else:
            print(f"Generating plot for HelloSimple-AI: {source_dropdown.value}")
            analysis = HelloSimpleAIAnalysis(source=source_dropdown.value)
            analysis.run_analysis()

dropdown.observe(on_source_change, names='value')
button.on_click(on_click_action)

grid = widgets.GridspecLayout(5, 1, grid_gap='10px')
grid[0, 0] = dropdown
grid[1, 0] = source_dropdown
grid[2, 0] = lang_select
grid[3, 0] = lang_help
grid[4, 0] = button

grid.layout.width = '400px'
grid.layout.margin = '20px'

display(grid, output)